# nb_coord_silver_master_unify
## Coordinator: Silver Cleansed -> Silver Unified Master

Orchestrates unification of master entities (Provider, Organisation, Contact).  
Each master entity has a **custom notebook** because of entity-specific  
survivorship rules, matching logic, and SCD2 merge strategies.

This coordinator calls custom notebooks via `notebookutils.notebook.run`  
for each master entity (since custom logic prevents generic %run approach).



In [1]:
%run "/05 - shared-libraries/nb_shared_transforms"

StatementMeta(DataWHv35, 8051, -1, Finished, Available, Finished, False)

In [2]:
%run "/05 - shared-libraries/nb_shared_logging"

StatementMeta(DataWHv35, 8051, -1, Finished, Available, Finished, False)

In [3]:
# Pipeline parameters
storage_account_name = ""
batch_run_id = ""
run_date = ""
retry_count = 0
pipeline_id = ""
parent_pipeline_id = ""
top_level_pipeline_id = ""
pipeline_name =""

StatementMeta(DataWHv35, 8051, 19, Finished, Available, Finished, False)

In [4]:
# This cell is generated from runtime parameters. Learn more: https://go.microsoft.com/fwlink/?linkid=2161015
batch_run_id = "run_20261126T0542"
run_date = "20261126T0542"
pipeline_id = "7260fa8e-10d2-49ac-b4f5-b59911377a87"
parent_pipeline_id = "28cefcb0-0426-4e15-82da-b21ca1b49083"
top_level_pipeline_id = "0097f112-230e-4e76-adb6-5e9af5475185"
storage_account_name = "deintdlsaueaasqebla"
pipeline_name = "pl_coordinator_silver_master_unify"
retry_count = 0


StatementMeta(DataWHv35, 8051, 20, Finished, Available, Finished, False)

In [5]:
from mdw_shared import audit, metadata, transforms
from mdw_shared.audit import *
from mdw_shared.metadata import *
from mdw_shared.transforms import *

from pyspark.sql import SparkSession
import json as _json
from pyspark.sql.functions import (
    col, lit, trim, upper, coalesce, when, row_number, current_timestamp
)
import os
from concurrent.futures import ThreadPoolExecutor, as_completed


# Get spark session and delta lake paths
spark = get_spark_session()

data_lake_paths = get_datalake_paths(storage_account_name)

BRONZE_BASE_PATH = data_lake_paths["bronze"]
SILVER_BASE_PATH = data_lake_paths["silver"]
GOLD_BASE_PATH = data_lake_paths["gold"]
METADATA_BASE_PATH = data_lake_paths["metadata"]

METADATA_CONTROL_PATH = f"{METADATA_BASE_PATH}/control"
METADATA_AUDIT_PATH = f"{METADATA_BASE_PATH}/audit"
METADATA_CONFIG_PATH = f"{METADATA_BASE_PATH}/config"

STEP_NAME_TO_RUN = "silver_master_unify" 
PIPELINE_NAME = pipeline_name
NOTEBOOK_NAME = "nb_coord_silver_master_unify"

REQUIRED_PIPELINE_NAME = "pl_coordinator_silver_arms_ingest"

StatementMeta(DataWHv35, 8051, 21, Finished, Available, Finished, False)

## Parse Notebook Results

In [6]:
def _parse_notebook_result(result):
    """Parse JSON metrics and mastering fields from child notebook exit value."""
    rows_read, rows_written = 0, 0
    mastering_systems, matched_on, surv_strategy, target_path_override = None, None, None, None
    if result:               
        metrics = _json.loads(str(result))
        rows_read = metrics.get("rows_read", 0)
        rows_written = metrics.get("rows_written", 0)
        mastering_systems = metrics.get("mastering_source_systems_involved")
        matched_on = metrics.get("matched_on")
        surv_strategy = metrics.get("survivorship_strategy")
        target_path_override = metrics.get("target_path_for_lineage") or metrics.get("target_path")

    return rows_read, rows_written, mastering_systems, matched_on, surv_strategy, target_path_override

StatementMeta(DataWHv35, 8051, 22, Finished, Available, Finished, False)

## Function to run master entities 

In [7]:
# Processing order:
# 1. If notebook_to_execute is defined, execute notebook.
# 2. Otherwise execute JSON-config-based unification.
def _run_master_entity(cfg):
    entity_name = cfg.target_entity_name
    source_entity = cfg.source_entity_name
    target_path = cfg.target_path.lower()
    source_path = cfg.source_path.lower()
    notebook_path = cfg.notebook_to_execute
    json_config = cfg.json_config

    try:
        act_id, _act_start = log_activity_started(METADATA_AUDIT_PATH,
            batch_run_id = batch_run_id, 
            activity_name = f"unify_master_{entity_name}",
            notebook_name = NOTEBOOK_NAME, 
            source_entity_name = source_entity,
            target_entity_name = entity_name, 
            source_layer = EVENT_PHASE_CLEANSE,
            target_layer = EVENT_PHASE_UNIFY, 
            retry_count = retry_count,
            pipeline_id = pipeline_id
        )
            
        # Run customized notebook if note_to_execute path exist
        if notebook_path:
            print(f"notebook_path :{notebook_path}")
            result = notebookutils.notebook.run(
                notebook_path,
                timeout_seconds=3600,
                arguments={
                    "storage_account_name": storage_account_name,
                    "batch_run_id": batch_run_id,
                    "run_date": run_date,
                    "activity_run_id": act_id,
                    "retry_count": retry_count,
                    "pipeline_name": pipeline_name,
                    "source_entity_name": source_entity,
                    "target_entity_name": entity_name,
                    "source_path": source_path,
                    "target_path": target_path,
                    "json_config": json_config
                }
            )

            rows_read, rows_written, _nb_mastering, _nb_matched_on, _nb_surv_strategy, _nb_target_path = _parse_notebook_result(result)        
            target_path = _nb_target_path or target_path
        
        else:
            if not source_entity :
                raise Exception("Requires either source_entity or notebook_path")

            df_target = (spark.read.format("delta")
                .load(source_path)
                .filter(col("mdw_batch_run_id") == batch_run_id)
                )

            df_target,primary_key_col_name_arr = apply_master_entity_rules(
                df = df_target,
                entity_name = entity_name,
                metadata_config_path = f"{METADATA_CONFIG_PATH}/{UNIFIED_MASTER_JSON_CONFIG_FOLDERNAME}",
                json_config = cfg.json_config,
                batch_run_id = batch_run_id,
                act_id = act_id,
                source_system = ARMS_SOURCE_SYSTEM )

            rows_new, rows_changed = scd2_merge(
                spark = spark,
                source_df = df_target,
                target_path = target_path,
                key_columns = primary_key_col_name_arr,
                scd_columns = None,  # compare all non-metadata columns
                batch_run_id = batch_run_id,
                activity_run_id = act_id
            )
            
            rows_read = rows_new + rows_changed
            rows_written = rows_new + rows_changed
        
            # Capture Delta version after child notebook wrote to target
            target_version = capture_delta_version(spark, target_path)

            # Log lineage event (source -> target)
            source_version = capture_delta_version(spark, source_path) if source_path else None
            log_lineage_event(METADATA_AUDIT_PATH,
                batch_run_id=batch_run_id,
                source_entity=source_entity, 
                source_layer=EVENT_PHASE_CLEANSE,
                target_entity=entity_name, 
                target_layer=EVENT_PHASE_UNIFY,
                activity_run_id=act_id,
                source_path=source_path, 
                target_path=target_path,
                transform_type=EVENT_PHASE_UNIFY_MASTER,
                delta_version_source=source_version,
                delta_version_target=target_version,
                notebook_name=notebook_path or NOTEBOOK_NAME, 
                pipeline_name=PIPELINE_NAME,
                rows_in=rows_read, 
                rows_out=rows_written
            )

        log_activity_completed(METADATA_AUDIT_PATH,
            activity_run_id=act_id, 
            batch_run_id=batch_run_id,
            activity_name=f"unify_master_{entity_name}", 
            status=EVENT_STATUS_SUCCEEDED,
            notebook_name=notebook_path or NOTEBOOK_NAME, 
            source_entity_name=source_entity,
            target_entity_name=entity_name, 
            source_layer=EVENT_PHASE_CLEANSE,
            target_layer=EVENT_PHASE_UNIFY, 
            retry_count=retry_count,
            pipeline_id=pipeline_id,
            started_at=_act_start
        )
                
        return None

    except Exception as e:        
        err_msg = f"Silver master unification has failed for entity: {entity_name}"
        
        log_activity_completed(METADATA_AUDIT_PATH,
            activity_run_id=act_id, 
            batch_run_id=batch_run_id,
            activity_name=f"unify_master_{entity_name}", 
            status=EVENT_STATUS_FAILED,
            notebook_name=notebook_path or NOTEBOOK_NAME, 
            error_message=format_exception_message(e, err_msg), 
            retry_count=retry_count,
            pipeline_id=pipeline_id,
            started_at=_act_start
        )        
        raise Exception(err_msg) from e

StatementMeta(DataWHv35, 8051, 23, Finished, Available, Finished, False)

## Get unification configuration and process it

In [8]:
all_succeeded = True
error_messages = []

# store exception as it will be used at the end of the notebook
captured_exception = None
try:
    log_batch_event(METADATA_AUDIT_PATH,
        batch_run_id=batch_run_id, 
        run_date=run_date, 
        retry_count=retry_count,
        event_type=EVENT_TYPE_STARTED, 
        status=EVENT_STATUS_RUNNING, 
        pipeline_name=PIPELINE_NAME,
        pipeline_id=pipeline_id, 
        parent_pipeline_id=parent_pipeline_id,
        top_level_pipeline_id=top_level_pipeline_id,
        phase=EVENT_PHASE_UNIFY_MASTER 
    )

    # Promotion check: verify Silver Cleanse completed successfully before unification
    assert_batch_promotion(METADATA_AUDIT_PATH, batch_run_id,
        required_pipeline_name=REQUIRED_PIPELINE_NAME,
        current_phase=EVENT_PHASE_UNIFY_MASTER
    )
        
    # Discover master unification entities from metadata
    transform_configs = get_transform_config(METADATA_CONTROL_PATH, STEP_NAME_TO_RUN)    

    if not transform_configs:
        raise Exception(
                f"Transform configs not found for Metadata Control Path: "
                f"{METADATA_CONTROL_PATH}, Step Name to run {STEP_NAME_TO_RUN}"
            )

    for cfg in transform_configs:
        result = _run_master_entity(cfg)                

except Exception as e:
    e_msg = format_exception_message(e, str(e))
    captured_exception = Exception(e_msg)

# =============================================================================
# Log batch completed
# =============================================================================
final_status = EVENT_STATUS_SUCCEEDED
errormessage_str=None
message = f"Silver master unification has {final_status} for batch_run_id: {batch_run_id}"

if captured_exception is not None:
    final_status = EVENT_STATUS_FAILED
    message = f"Silver master unification has {final_status} for batch_run_id: {batch_run_id}"
    errormessage_str = format_exception_message(captured_exception, message)

log_batch_event(METADATA_AUDIT_PATH,
    batch_run_id=batch_run_id,
    run_date=run_date,
    retry_count=retry_count,
    event_type=EVENT_TYPE_COMPLETED,
    status=final_status,
    pipeline_name=PIPELINE_NAME,
    pipeline_id=pipeline_id,
    parent_pipeline_id=parent_pipeline_id,
    top_level_pipeline_id=top_level_pipeline_id,
    phase=EVENT_PHASE_UNIFY_MASTER,
    error_message=errormessage_str
)

print(message)

if captured_exception is not None:
    raise Exception(message) from captured_exception 

StatementMeta(DataWHv35, 8051, 24, Finished, Available, Finished, False)

2026-11-26T06:16:10.671Z | INFO    | mdw_shared.audit | batch=- | pipeline=- | Batch event: started running pipeline=pl_coordinator_silver_master_unify batch=run_20261126T0542
2026-11-26T06:16:20.219Z | INFO    | mdw_shared.audit | batch=- | pipeline=- | Batch event: completed failed pipeline=pl_coordinator_silver_master_unify batch=run_20261126T0542
Silver master unification has failed for batch_run_id: run_20261126T0542


Exception: Silver master unification has failed for batch_run_id: run_20261126T0542